In [76]:
import os 
import MDAnalysis as mda
import numpy as np
from MDAnalysis import transformations
base_dir = "/biggin/b212/bioc1781/Projects/COPI/model_complex/trimer/martini_type1_with_kdelr/pdbs/"
#make universe out of "type1_em_segs.pdb"

# align the two
ref = mda.Universe(base_dir + "type1_em_segs_EMBED.pdb")
u = mda.Universe(base_dir + "kdelr_af2_rotated.pdb")
# select all "DUM"  atoms in ref
ref_sel = ref.select_atoms("resname DUM")

# select all "DUM"  atoms in ref
ref_sel = ref.select_atoms("name CA")
ref_sel_prot = ref.select_atoms("protein")

# get the distance between the COMs of ref_sel and u
d = ref_sel_prot.center_of_mass() - u.atoms.center_of_mass()
# translate u by this amount
u.atoms.translate(d)


# get average z position of DUM atoms
z_ref = ref_sel.center_of_mass()[2]

#translate u to average z position of DUM atoms
u.atoms.translate([0,0,-z_ref])

# spacing between KDELR molecules
spacing = 180

n = 1
#  translate copies of kdelr in x by spacing n times
for i in range(-n, n+1):
    u.atoms.translate([spacing*i,0,0])
    # translate copies of kdelr in y by spacing n times
    for j in range(-n, n+1):
        u.atoms.translate([0,spacing*j,0])

        # random rotation about z
        ts = u.trajectory.ts    
        angle = np.random.rand()*360
        ag = u.atoms
        d = [0,0,1]
        rotated = transformations.rotate.rotateby(angle, direction=d, ag=ag)(ts)

        # write out the aligned structures
        u.atoms.write(base_dir + "kdelr_af2_aligned_to_membrane_" + str(i) + "_" + str(j) + ".pdb")
        #ref.atoms.write(base_dir + "type1_em_segs_EMBED_aligned_" + str(i) + "_" + str(j) + ".pdb")
        u.atoms.translate([0,-spacing*j,0])
    u.atoms.translate([-spacing*i,0,0])

# make a universe with all of these copies combined using merge
    
#u_all = ref

chainID = ord('A')
for i in range(-n, n+1):
    for j in range(-n, n+1):
        u_copy = mda.Universe(base_dir + "kdelr_af2_aligned_to_membrane_" + str(i) + "_" + str(j) + ".pdb")
        # change the chain ID
        for a in u_copy.atoms:
            a.chainID = chr(chainID)
        if i == -n and j == -n:
            u_all = ref
            u_all = mda.Merge(u_all.atoms, u_copy.atoms)
        else:
            u_all = mda.Merge(u_all.atoms, u_copy.atoms)
        chainID += 1

# /dev/sda1

# select everythign except residues "DUM" in u_all
u_all_sel = u_all.select_atoms("not resname DUM")


# write out the aligned structures
u.atoms.write(base_dir + "kdelr_af2_aligned_to_membrane.pdb")
ref.atoms.write(base_dir + "type1_em_segs_EMBED_aligned.pdb")
u_all_sel.atoms.write(base_dir + "cop1+kdelr_aligned_to_membrane.pdb")

# read in the u_all pdb and write it out line by line, adding a ter statement if there is a chainID change
with open(base_dir + "cop1+kdelr_aligned_to_membrane.pdb", "r") as f:
    lines = f.readlines()
with open(base_dir + "cop1+kdelr_aligned_to_membrane.pdb", "w") as f:
    chainID = "A"
    for line in lines:
        if line[0:4] == "ATOM":
            if line[21] != chainID:
                f.write("TER\n")
                chainID = line[21]
        f.write(line)
    f.write("TER\n")
    f.write("END\n")





/biggin/b212/bioc1781/.local/lib/python3.10/site-packages/MDAnalysis/topology/PDBParser.py:331: UserWarning: Element information is missing, elements attribute will not be populated. If needed these can be guessed using MDAnalysis.topology.guessers.
  warnings.warn("Element information is missing, elements attribute "
/biggin/b212/bioc1781/.local/lib/python3.10/site-packages/MDAnalysis/topology/PDBParser.py:348: UserWarning: Unknown element  found for some atoms. These have been given an empty element record. If needed they can be guessed using MDAnalysis.topology.guessers.
  warnings.warn(wmsg)
/biggin/b212/bioc1781/.local/lib/python3.10/site-packages/MDAnalysis/topology/guessers.py:146: UserWarning: Failed to guess the mass for the following atom types: 
  warnings.warn("Failed to guess the mass for the following atom types: {}".format(atom_type))
/biggin/b212/bioc1781/.local/lib/python3.10/site-packages/MDAnalysis/coordinates/PDB.py:451: UserWarning: 1 A^3 CRYST1 record, this is usu